[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb05_task_der.ipynb)

# NB05 — Task 2: DER Arbitrage

> **Prerequisites**: [NB04 — Task 1: OPF](./nb04_task_opf.ipynb)  
> **Time**: ~10 minutes

## What You'll Learn

1. Why Task 2 is **harder** than Task 1 from an RL perspective (temporal dependency)
2. The SOC (State of Charge) as a **persistent state** across timesteps
3. How peak–valley price arbitrage works as an RL objective
4. SOC trajectory visualization for 3 cooperative battery agents

## Background: How is Task 2 Different from Task 1?

| Aspect | Task 1 (OPF) | Task 2 (DER Arbitrage) |
|---|---|---|
| Grid type | Transmission (5-bus) | Distribution (33-bus) |
| Agent controls | Generator output score | Battery charge/discharge power |
| State dependency | Memoryless (each step independent) | **SOC persists across steps** |
| Constraint type | Line flow limits | SOC bounds + node voltage limits |
| Optimal strategy | Proportional cost minimization | **Intertemporal arbitrage planning** |
| Credit assignment | Shared cost | Shared profit |

**Key ML insight**: Task 2 is a **POMDP with persistent state**. The SOC at step $t$
depends on all previous charge/discharge decisions. This breaks the "each step is
independent" assumption that makes Task 1 easier.

**Arbitrage in one sentence**: charge (buy energy) when electricity is cheap (night),
discharge (sell energy back to grid) when it is expensive (peak hours). The battery's
SOC is the "account balance" that must be managed across the whole episode.

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper
from powerzoo.benchmarks.policies import RandomPolicy, RuleBasedPolicy
from powerzoo.benchmarks import evaluate

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Task Configuration

In [ ]:
from powerzoo.tasks.simple.task2_marl_der_arbitrage import MARLDERArbitrageTask

task = MARLDERArbitrageTask(split='train')
scenario = task.get_scenario_config()

print("=== Task 2: marl_der_arbitrage ===")
print(f"Grid:         Case33bw — IEEE 33-bus radial distribution, 12.66 kV, ~3.7 MW")
print(f"Num agents:   {task._num_batteries} batteries")
print(f"Battery spec: {task._battery_capacity_mwh*1000:.0f} kWh / {task._battery_power_mw*1000:.0f} kW each")
print(f"Bus IDs:      {task._battery_bus_ids}  (spread along main feeder trunk)")
print(f"SOC range:    [{task._soc_min}, {task._soc_max}]")
print(f"\nReward:")
reward_cfg = scenario['reward']
print(f"  Peak hours:     {reward_cfg['peak_hours']}")
print(f"  Off-peak hours: {reward_cfg['off_peak_hours']}")
print(f"  Arbitrage weight:    {reward_cfg['arbitrage_weight']}")
print(f"  SOC penalty weight:  {reward_cfg['soc_penalty_weight']}")

## 2. The Persistent SOC State

Unlike Task 1 where each step is nearly independent, here the SOC at each step
depends on the entire history of charge/discharge decisions.

The observation includes each battery's current SOC — this is the key persistent state.

In [ ]:
env = make_task_env("marl_der_arbitrage", split="train")
gym_env = GymnasiumWrapper(env)

obs, info = gym_env.reset(seed=42)
print(f"Obs shape: {obs.shape}")
print(f"Act shape: {gym_env.action_space.shape}")
print(f"Action range: [{gym_env.action_space.low[0]:.2f}, {gym_env.action_space.high[0]:.2f}] MW")
print("  Positive = charge (absorb power from grid)")
print("  Negative = discharge (inject power to grid, V2G)")

## 3. Price Signal and Peak-Valley Pattern

In [ ]:
# Visualize the peak / off-peak price pattern (binary signal from task config)
peak_hours = set(reward_cfg['peak_hours'])
off_peak_hours = set(reward_cfg['off_peak_hours'])

hours = np.arange(24)
price = np.array([2.0 if h in peak_hours else 0.5 for h in hours])

fig, ax = plt.subplots(figsize=(10, 3))
ax.step(hours, price, where='post', color='darkorange', linewidth=2)
ax.fill_between(hours, price, step='post', alpha=0.25, color='darkorange')
ax.axhline(1.0, color='gray', linewidth=0.8, linestyle='--', label='Break-even')

# Mark peak regions
for h in peak_hours:
    ax.axvspan(h, h+1, alpha=0.07, color='red')

ax.set_xticks(range(0, 25, 2))
ax.set_xlabel('Hour of day')
ax.set_ylabel('Price (relative units)')
ax.set_title('Task 2 price signal — peak (red shading) vs off-peak (blue) hours')
ax.legend()
plt.tight_layout()
plt.show()
print("Optimal strategy: charge during off-peak, discharge during peak.")

## 4. SOC Trajectory: Rule-Based vs Random Policy

In [ ]:
def collect_episode_data(task_name, policy_fn, seed=42):
    """Run one episode collecting (step, reward, soc_per_agent) data."""
    raw_env = make_task_env(task_name, split='train')
    obs_raw, info = raw_env.reset(seed=seed)

    rewards = []
    soc_history = {agent: [] for agent in raw_env.agents}

    while raw_env.agents:
        # Build action dict for all current agents
        actions = {}
        for agent in raw_env.agents:
            agent_obs = obs_raw[agent]
            actions[agent] = policy_fn(agent, agent_obs, raw_env)

        obs_raw, reward_dict, term_dict, trunc_dict, info = raw_env.step(actions)
        rewards.append(sum(reward_dict.values()))

        # Collect SOC from obs (SOC is in local obs, last dimension before limits)
        for agent, agent_obs in obs_raw.items():
            # SOC is typically the first local obs dimension
            soc_history[agent].append(float(agent_obs[0]) if len(agent_obs) > 0 else 0.5)

    return rewards, soc_history

# Random policy
raw_env_ref = make_task_env("marl_der_arbitrage", split='train')
def random_policy_fn(agent, obs, env):
    return env.action_space[agent].sample()

# Rule-based: charge when hour < 6 or hour > 21, discharge 9-20
def rule_policy_fn(agent, obs, env):
    act_space = env.action_space[agent]
    p_max = act_space.high[0]
    # Simple rule: alternate charge/discharge based on obs time signal
    # time_cos > 0 → first half of day (charge), else discharge
    if len(obs) >= 3 and obs[2] > 0:  # approximate peak indicator
        return np.array([-p_max * 0.8] * act_space.shape[0])  # discharge
    else:
        return np.array([p_max * 0.8] * act_space.shape[0])   # charge

random_rewards, random_soc = collect_episode_data("marl_der_arbitrage", random_policy_fn)
rule_rewards,   rule_soc   = collect_episode_data("marl_der_arbitrage", rule_policy_fn)

print(f"Random policy total reward: {sum(random_rewards):.2f}")
print(f"Rule-based total reward:    {sum(rule_rewards):.2f}")

In [ ]:
# Plot SOC trajectories for both policies
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

colors = ['#1565C0', '#E65100', '#2E7D32']

for ax, (title, soc_data) in zip(axes, [
    ("Random policy", random_soc),
    ("Rule-based policy", rule_soc)
]):
    for i, (agent, soc_vals) in enumerate(soc_data.items()):
        ax.plot(soc_vals, color=colors[i % len(colors)], linewidth=1.8,
                label=f'{agent} (bus {[6,12,18][i % 3]})')
    ax.axhline(0.1, color='red', linewidth=1, linestyle=':', alpha=0.6, label='SOC_min=0.1')
    ax.axhline(0.9, color='red', linewidth=1, linestyle=':', alpha=0.6, label='SOC_max=0.9')
    ax.set_ylabel('State of Charge (SOC)')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8, ncol=2)

axes[1].set_xlabel('Timestep (30-min interval)')
plt.suptitle('Task 2 — SOC trajectories: 3 battery agents over one episode', y=1.01)
plt.tight_layout()
plt.show()

print("\nNote: random policy charges/discharges erratically → poor arbitrage.")
print("A well-trained policy should charge during off-peak and discharge at peak.")

## 5. Why Temporal Planning Matters

The SOC constraint creates a **balance constraint** over the episode:

$$\text{SOC}_{t+1} = \text{SOC}_t + \frac{\eta \cdot p_t \cdot \Delta t}{E_{\text{cap}}}$$

where $p_t$ is charge power (positive) or discharge power (negative), $\eta = 0.95$ is
round-trip efficiency, and $E_{\text{cap}} = 500$ kWh.

**ML analogy**: this is like a memory cell in an LSTM — the agent must maintain a
"balance" across timesteps. Greedy per-step policies fail because they deplete the
battery before peak hours arrive.

In [ ]:
# Simulate the SOC dynamics manually to illustrate
soc = 0.5          # initial SOC
capacity_mwh = 0.5 # 500 kWh
p_max = 0.2        # 200 kW
eta = 0.95
dt = 0.5           # 30-min = 0.5 hours

# Optimal schedule: charge 0:00-8:00, discharge 9:00-20:00
optimal_power = []
for h in range(24):
    for step in range(2):  # 2 steps per hour at 30 min
        if h < 8:
            optimal_power.append(p_max)    # charge
        elif 9 <= h <= 20:
            optimal_power.append(-p_max)   # discharge
        else:
            optimal_power.append(0.0)       # idle

soc_trace = [soc]
for p in optimal_power[:47]:
    delta = eta * p * dt / capacity_mwh if p > 0 else p * dt / capacity_mwh
    soc = np.clip(soc + delta, 0.1, 0.9)
    soc_trace.append(soc)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ax1.step(range(len(optimal_power[:48])), optimal_power[:48], where='post',
         color='steelblue', linewidth=2)
ax1.axhline(0, color='gray', linewidth=0.8)
ax1.set_ylabel('Charge power (MW)')
ax1.set_title('Optimal schedule (conceptual): charge cheap, discharge peak')

ax2.plot(soc_trace, color='darkorange', linewidth=2)
ax2.axhline(0.1, color='red', linestyle=':', linewidth=1, label='SOC bounds')
ax2.axhline(0.9, color='red', linestyle=':', linewidth=1)
ax2.set_ylabel('SOC')
ax2.set_ylim(0, 1.05)
ax2.set_xlabel('Timestep (30-min)')
ax2.legend()

plt.tight_layout()
plt.show()

---

## Summary — Key Differences from Task 1

| Aspect | Task 1 (OPF) | Task 2 (DER) |
|---|---|---|
| **State dependency** | Memoryless (each step ≈ independent) | **SOC persists: history matters** |
| **Optimal strategy** | Minimize instantaneous cost | **Plan across the full episode** |
| **Constraint type** | Line flow limits (soft penalty) | SOC bounds + voltage (hard + soft) |
| **Grid type** | Transmission (5-bus) | Distribution (33-bus radial) |
| **RL challenge** | Cooperative cost sharing | Intertemporal planning + coordination |

## Next

→ [NB06 — Task 3: EV V2G (The Availability Mask)](./nb06_task_ev.ipynb)